# 15.06 · Jev as a LangSmith evaluator vs LLM-as-judge

**Module 15 · Decision Models with Jev** · Part 2 (Pro) · optional

| | |
|---|---|
| ⏱️ Estimated time | 2.5 h |
| 📋 Prerequisites | Lessons 15.01–15.02; Module 10 (datasets, experiments, LLM-as-judge, judge agreement). |
| 🔑 Keys | optional `TYPESAFE_API_KEY` · optional `GROQ_API_KEY` (LLM judge, 24 calls) · optional `LANGSMITH_API_KEY` (upload the experiment) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (no network, free) · **⚖️** Jev: live with `TYPESAFE_API_KEY` (paid, fractions of a cent), otherwise replayed from recordings · **💸** calls Groq (uses your free-tier quota).
> Without keys, ⚖️ cells replay recorded answers, 💸 cells print `⏭️ Skipped`, and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Write** a typed evaluator with Jev: one request, several scored questions, in the `{"key", "score"}` shape LangSmith expects.
2. **Validate** it against human labels with Module 10's tools (TPR / TNR with intervals, calibration) before trusting it.
3. **Pick** the evaluator's threshold on a dev split from what a miss and a false alarm cost, and check it on a test split.
4. **Compare** it with an LLM-as-judge on the same items, the same code path and the same metrics, and decide which belongs where.

## 📚 Official reading (verified 2026-10-16)

| Page | Read it for |
|---|---|
| [LangSmith: evaluation concepts](https://docs.langchain.com/langsmith/evaluation-concepts) | Datasets, experiments, evaluators, feedback keys |
| [LangSmith: define a custom code evaluator](https://docs.langchain.com/langsmith/code-evaluator) | The `(inputs, outputs, reference_outputs)` signature and return shape |
| [TypeSafe docs: questions](https://docs.typesafe.ai) | `Noul` instructions, what a probability means |
| Module 10, Lessons on LLM-as-judge | Judge agreement, Rogan-Gladen correction, thresholds |

> **Why this matters at work.** Orbita's support bot drafts thousands of replies a day. Grading each one with a frontier LLM judge costs more than the bot. A judge that says "pass" with no probability can't be tuned, so every product manager argues about its strictness. Jev returns a **probability per question**: you can set the strictness on labeled data, change it without re-prompting, and run it on every reply for fractions of a cent.

> 🧪 **The data.** 48 support replies (8 tickets × 6 reply kinds: `resolves`, `brief`, `vague`, `invents`, `off_topic`, `deflects`) with a human label (`fails`). The `brief` and `invents` replies are the hard ones. Tickets 1–4 are the **dev** split and 5–8 the **test** split. The data is synthetic, and so are the replayed Jev answers: the lesson teaches the method.

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-typesafe[experimental]==0.0.1a3" "langchain-groq>=1.1.3,<2" "langsmith>=0.14.11,<1"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-typesafe", "langsmith",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/15-decision-models-with-jev/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('TYPESAFE_API_KEY', 'GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm15-jev-as-evaluator'
# Setup 3: Load .env, check keys WITHOUT printing them, and build Jev in live or replay mode.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.decisions import decision_classifier
from mastery.doctor import check_decisions_env

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_TYPESAFE_KEY = is_set("TYPESAFE_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m15-<lesson>: one tracing project per lesson
JEV = decision_classifier()                                      # live with a key, replay without (never raises offline)
jev = JEV.classifier

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast, "(Groq)")
print("jev       :", JEV.describe())
for note in check_decisions_env():
    print("⚠️", note)
if JEV.mode == "replay":
    print("\nℹ️ Replay mode: ⚖️ cells answer from recorded Jev responses (marked synthetic until real ones are recorded).")
    print("   They show the API and the decision logic; they are NOT measurements of Jev. Add TYPESAFE_API_KEY to go live.")
if not HAVE_GROQ_KEY:
    print("⚠️ No GROQ_API_KEY: 💸 Groq cells will print ⏭️ Skipped; 🧪 and ⚖️ cells still run.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# Preview models warn once per model build; the lesson says so in Markdown already.
from mastery.models import PreviewModelWarning
warnings.filterwarnings("once", category=PreviewModelWarning)
# langchain-typesafe 0.0.1a3 marks TypeSafeClassifier as beta on every construction; the lesson says so.
warnings.filterwarnings("ignore", message=".*TypeSafeClassifier.*beta")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

from collections import Counter

from mastery.decisions import provenance
from mastery.decisions.evaluator import KINDS, POLICY, REPLY_CASES, bot_under_test, eval_questions, examples, jev_evaluators

print(len(REPLY_CASES), "replies ·", dict(Counter(c["split"] for c in REPLY_CASES)), "· failing:", sum(c["fails"] for c in REPLY_CASES))
case = next(c for c in REPLY_CASES if c["id"] == "R1-invents")
print("\nticket:", case["ticket"]); print("reply :", case["reply"]); print("label : fails =", case["fails"])
print("\nquestions:", {k: q.instructions[:70] + "…" for k, q in eval_questions().items()})

**You should now see** 48 replies (24 dev, 24 test), 32 of them labeled as failing. The example reply sounds helpful but **invents** a policy ("3 free months as our standard apology"), so its label is `fails = True`. Jev gets two questions: `fails` (the verdict) and `invents_policy` (a diagnostic that says *why*).

---
## Part 1 · An evaluator in LangSmith's shape

> ⚖️ **Jev cell.** With `TYPESAFE_API_KEY` it calls TypeSafe (billed at $0.042 per 1M input tokens, a fraction of a cent here). Without a key it **replays** recorded answers: same API, same objects, no network. Until real answers are recorded, replayed answers are *synthetic* (hand-written from the documented schema): learn the API and the logic from them, not Jev's accuracy.

In [ ]:
# ⚖️ Run the offline twin of langsmith.evaluate: target = the bot under test, evaluators = Jev (one request per reply).
from mastery.evals.datasets import run_experiment

jev_fails, jev_invents = jev_evaluators(jev)
print(jev_fails(inputs={"ticket": case["ticket"]}, outputs={"reply": case["reply"]}))

dev = run_experiment(bot_under_test, examples("dev"), [jev_fails, jev_invents], name="jev-grader-dev")
rows = []
for kind in KINDS:
    ids = [e["id"] for e in examples("dev") if e["reference"]["kind"] == kind]
    mean = lambda key: sum(dev.runs[0][i]["scores"][key] for i in ids) / len(ids)
    rows.append({"reply kind": kind, "label": "fails" if REPLY_CASES[KINDS.index(kind)]["fails"] else "passes",
                 "mean p(fails)": f"{mean('fails'):.2f}", "mean p(invents_policy)": f"{mean('invents_policy'):.2f}"})
table(rows, ["reply kind", "label", "mean p(fails)", "mean p(invents_policy)"])
print("\nJev answers:", provenance(jev.invoke({"state": {"policy": POLICY, "customer_ticket": case["ticket"],
                                                           "support_reply": case["reply"]}, "questions": eval_questions()})))

**You should now see** a single evaluator result like `{'key': 'fails', 'score': 0.6…, 'comment': 'jev jev-1.13.0'}`, then a table by reply kind. `off_topic`, `vague` and `deflects` score high on `p(fails)` and `resolves` scores low. `brief` (fine, but terse) and `invents` (fluent, but wrong) sit in the middle. Only `invents` scores high on `invents_policy`.

- The two evaluators share **one** Jev request per reply (a small cache in `jev_evaluators`), so a second feedback key is free. With an LLM judge each key usually means another prompt, or a longer one.
- The return value is plain `{"key", "score"}`, so the same functions go to `langsmith.evaluate(...)` unchanged (Part 4).

---
## Part 2 · Trust, then threshold: validate on dev, check on test

In [ ]:
# 🧪 Agreement with human labels at 0.5, then a threshold picked on DEV from costs (a miss costs 3 false alarms).
from mastery.evals.judges import agreement, choose_threshold, expected_calibration_error

def scores(exp, split, key="fails"):
    ex = examples(split)
    return [exp.runs[0][e["id"]]["scores"][key] for e in ex], [e["reference"]["fails"] for e in ex]

s_dev, y_dev = scores(dev, "dev")
print("dev @0.50 :", agreement([s >= 0.5 for s in s_dev], y_dev))
pick = choose_threshold(s_dev, y_dev, cost_false_alarm=1, cost_miss=3)
t = pick["threshold"]
print(f"chosen on dev: t = {t} (cost {pick['cost']})")

test = run_experiment(bot_under_test, examples("test"), [jev_fails, jev_invents], name="jev-grader-test")
s_test, y_test = scores(test, "test")
print(f"test @{t:.2f} :", agreement([s >= t for s in s_test], y_test))
conf = [max(s, 1 - s) for s in s_test]
print(f"test ECE (5 bins): {expected_calibration_error(conf, [(s >= 0.5) == y for s, y in zip(s_test, y_test)]):.3f}")

**You should now see:**

- **Dev at 0.5:** TNR 100% but TPR 88%. Two failing replies (the `invents` kind) slip through as passes.
- **Chosen on dev:** `t = 0.3`. A miss costs three times a false alarm, so the evaluator should be stricter than 0.5.
- **Test at 0.3:** every failing reply is caught (TPR 100%), but TNR drops to about 62%. A few `brief` replies get flagged.

The intervals are wide: 24 items per split is enough to teach the method, not to sign off a grader. Module 10's rule still holds: report the interval, label more data before you ship.

- The **threshold is a product decision**, made on labeled data, and you change a number to change it, not a prompt. An LLM judge with a fixed "pass/fail" output can't be tuned this way.
- Pick on dev, report on test. Choosing *and* measuring on the same 24 items would overstate how well the threshold works.

---
## Part 3 · The same items through an LLM-as-judge

In [ ]:
# 💸 Groq's fast model as the judge, through llm_decider: SAME questions, SAME evaluator code, SAME metrics (24 calls, ~12K tokens).
from mastery.decisions import llm_decider

if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")
else:
    from mastery.models import get_model
    llm_judge = llm_decider(get_model("fast", temperature=0))
    llm_fails, _ = jev_evaluators(llm_judge)       # the evaluator code doesn't care who answers
    llm_test = run_experiment(bot_under_test, examples("test"), [llm_fails], name="llm-judge-test")
    s_llm, _ = scores(llm_test, "test")
    print("LLM judge @0.50:", agreement([s >= 0.5 for s in s_llm], y_test))
    print("Jev       @" + f"{t:.2f}:", agreement([s >= t for s in s_test], y_test))
    print("distinct LLM scores:", sorted(set(s_llm)), "← one-hot verdicts: no threshold to tune")

**You should now see** (with a key) the LLM judge's TPR / TNR next to Jev's, plus the distinct scores the LLM gave. Usually they are only `0.0` and `1.0`, or a few round numbers it made up. Expect the LLM to catch the `invents` replies well, since spotting a contradicted policy is reading comprehension, which LLMs are good at. The results vary with the model and the day.

- **Why one-hot matters:** with only 0/1 scores, every threshold between 0 and 1 gives the same verdicts, so "make it stricter" means rewriting the prompt and re-validating.
- **Cost:** the LLM judge's prompt carries the questions as a JSON schema plus a system prompt, and it's billed at an LLM's per-token price. Jev's request is billed at $0.042 per 1M input tokens. Lesson 15.07 measures both.

| Use | Jev evaluator | LLM-as-judge |
|---|---|---|
| Every production reply, online monitoring | ✅ cheap, fast, tunable threshold | ❌ cost and latency add up |
| Nuanced rubric, needs a written reason | ❌ a probability, no explanation | ✅ reasoning in the comment |
| Offline experiments on a few hundred items | ✅ both work; Jev for the gate, LLM for the comments | ✅ |
| Grader whose strictness the business will tune | ✅ move the threshold | ❌ rewrite the prompt |

---
## Part 4 · Upload the experiment to LangSmith

In [ ]:
# 🌐 With LANGSMITH_API_KEY: create (once) the dataset m15-reply-grading and run langsmith.evaluate with Jev's evaluators.
if not is_set("LANGSMITH_API_KEY"):
    print("⏭️ Skipped: LANGSMITH_API_KEY is not set. Part 1's run_experiment is the offline twin of this cell.")
else:
    from langsmith import Client, evaluate
    from langsmith.utils import LangSmithError

    client, name = Client(), "m15-reply-grading"
    try:
        if not client.has_dataset(dataset_name=name):
            ds = client.create_dataset(name, description="Module 15.06: support replies with human fails labels (synthetic).")
            client.create_examples(dataset_id=ds.id, examples=[{"inputs": e["inputs"], "outputs": e["reference"]} for e in examples()])
        results = evaluate(bot_under_test, data=name, evaluators=[jev_fails, jev_invents],
                           experiment_prefix=f"jev-grader-{JEV.mode}", max_concurrency=4)
    except (LangSmithError, ValueError) as exc:   # bad key, wrong region, offline: the lesson still completes
        print(f"⛔ LangSmith upload failed ({type(exc).__name__}): {str(exc)[:160]}")
    else:
        print("experiment:", results.experiment_name, "· open it in LangSmith → Datasets & Experiments →", name)

**You should now see**, without a LangSmith key, `⏭️ Skipped` (and `⛔ LangSmith upload failed` if the key is wrong). With one, an experiment named `jev-grader-replay-…` (or `-live-…`) appears under the dataset `m15-reply-grading`. It shows a `fails` and an `invents_policy` feedback column for each of the 48 rows. Sort by `fails` to see the middle band where the `brief` and `invents` replies sit.

- LangSmith passes `reference_outputs` (the example's `outputs`) to each evaluator. Ours ignore it, since the label is only used to *validate* the grader (Part 2), not to grade.
- In replay mode the Jev answers are synthetic. The experiment shows the wiring, not Jev's quality.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Feedback column missing in LangSmith | Evaluator returned a bare float or a wrong key | Return `{"key": ..., "score": ...}` |
| Two Jev requests per row | One evaluator per key, each calling Jev | Share one request (the cache in `jev_evaluators`) |
| Threshold looks great on dev, worse on test | Chosen and measured on the same items | Choose on dev, report on test |
| `ReplayMiss` grading your own bot's replies | New replies have no recording | Expected offline; set `TYPESAFE_API_KEY` |
| LLM judge scores all `0.0`/`1.0` | One-hot verdicts | Use it for comments; tune strictness on a probabilistic grader |

> ⚠️ **Production note: the grader drifts with your bot.** (Part 2)
> Labeled replies from last quarter don't contain this quarter's failure modes. Re-label a fresh sample monthly and re-check TPR / TNR before you trust a trend line.

## 🏋️ Exercises

1. **(Easy)** Measure the test split at `0.5` instead of the chosen threshold. Which reply kind is missed, and which errors disappear?
2. **(Medium)** Use `invents_policy` as a second signal: flag a reply if `fails ≥ 0.5` **or** `invents_policy ≥ 0.6`. What happens to test TPR and TNR?
3. **(Medium)** Apply Module 10's `corrected_rate` to Jev's flags at `0.3` on the test split (dev as the validation set) and compare it with the label rate (16/24).
4. **(Stretch, needs keys)** Run Part 3 and Part 4 live. Where do Jev and the LLM disagree, and who is right?

<details>
<summary><b>Show solutions</b></summary>

**1.** TPR falls to 81% (3 `invents` replies pass) and TNR rises to 100% (the `brief` false alarms disappear). One threshold can't fix both, because the two hard kinds overlap in `p(fails)`.

**2.** On the synthetic test split, TPR 100% **and** TNR 100%: `invents_policy` separates the `invents` replies that `fails` alone couldn't. A diagnostic question can sharpen a gate. Check it on fresh labels before you believe a perfect score from 24 items.

**3.** `corrected_rate([s >= 0.3 for s in s_test], [s >= 0.3 for s in s_dev], y_dev)` gives about 76% with an interval of roughly [50%, 92%]. That contains the true 67%; with 24 validation items the correction is honest about how little it knows.

**4.** Sort disagreements by Jev's `p(fails)`: they cluster in the middle band. Label them yourself; that's the data that improves the next threshold.
</details>

## 🔎 Docs drill (5 min)

What arguments can a LangSmith custom code evaluator accept, and what may it return?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open **LangSmith → Evaluation → Define a custom code evaluator** (linked above).
2. Arguments are matched **by name**: `inputs`, `outputs`, `reference_outputs` (and `run`/`example` for the full objects).
3. Return a `bool`/number, a `{"key", "score"}` dict, or a list of them. Ours return dicts so the feedback key is explicit.

</details>

## ✅ Checkpoint

- [ ] I wrote a Jev evaluator that returns `{"key", "score"}` and shares one request across keys.
- [ ] I validated it against labels (TPR / TNR with intervals) before using it.
- [ ] I picked its threshold on dev from costs and reported on test.
- [ ] I can say when an LLM-as-judge is the better grader, and when it isn't.
- [ ] Restart & Run All passes with no keys; outputs cleared before saving.

## 🔎 Research checkpoint

Check whether LangSmith or `langchain-typesafe` now ships a ready-made Jev evaluator, or online-evaluator support for TypeSafe. Log it in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- A Jev evaluator is a plain function in LangSmith's shape: one typed request, a probability per question.
- **Validate, then threshold:** agreement with labels on dev, a cost-based threshold, the result on test.
- LLM judges give explanations; Jev gives a tunable probability at a fraction of the cost. Production monitoring wants the second, debugging wants the first.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 15-07-benchmarking-decisions.ipynb** → [`15-07-benchmarking-decisions.ipynb`](15-07-benchmarking-decisions.ipynb).

[Module 15 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)